<a href="https://colab.research.google.com/github/eeeaaai/YL/blob/main/fedasync_versiyon_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Asynchronous Federated Learning Server (main.py)
"""

from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/colab_files/flower_async_main_deneme_1')
!pip install flwr

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.15.1
    Uninstalling typer-0.15.1:
      Successfully uninstalled typer-0.15.1


In [13]:
import flwr as fl
from async_client_versiyon_1 import AsyncClient
import threading
import time
import numpy as np

class DummyModel:
    """A simple dummy model to simulate training."""
    def __init__(self):
        self.weights = [np.random.randn(10, 10), np.random.randn(10)]  # Random model weights

    def get_weights(self):
        return self.weights

    def set_weights(self, new_weights):
        self.weights = new_weights

def start_client(client_id):
    """Function to start a single client instance."""
    model = DummyModel()
    client = AsyncClient(cid=client_id, model=model)
    fl.client.start_numpy_client(server_address="localhost:8080", client=client)

# Launch multiple clients in separate threads
num_clients = 2
threads = []

for i in range(num_clients):
    t = threading.Thread(target=start_client, args=(i,))
    t.start()
    threads.append(t)
    time.sleep(1)  # Small delay to avoid congestion

# Wait for all clients to finish
for t in threads:
    t.join()


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
	Instead, use `flwr.client.start_client()` by ensuring you first call the `.to_client()` method as shown below: 
	flwr.client.start_client(
		server_address='<IP>:<PORT>',
		client=FlowerClient().to_client(), # <-- where FlowerClient is of type flwr.client.NumPyClient object
	)
	Using `start_numpy_client()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
	Instead, use `flwr.client.start_client()` by ensuring you first call the `.to_client()` method as shown below: 
	flwr.client.start_client(
		server_address='<IP>:<PORT>',
		client=FlowerClient().to_

In [12]:
from async_server import AsyncServer
from async_client_manager import AsyncClientManager
from async_strategy import AsynchronousStrategy
from flwr.server.strategy import FedAvg

# Create AsyncServer instance
server = AsyncServer(
    strategy=FedAvg(),
    client_manager=AsyncClientManager(),
    async_strategy=AsynchronousStrategy(
        total_samples=1000,
        staleness_alpha=0.9,
        fedasync_mixing_alpha=0.5,
        fedasync_a=1.2,
        num_clients=2,
        async_aggregation_strategy="fedasync",
        use_staleness=True,
        use_sample_weighing=True,
        send_gradients=False,
        server_artificial_delay=False,
    ),
    base_conf_dict={},
)
print(f"Number of available clients: {server._client_manager.num_available()}")
print(f"List of clients: {server._client_manager.all()}")

print("🚀 Starting Asynchronous Federated Learning Server...")
history = server.fit(num_rounds=10, timeout=60)


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
INFO :      Initializing global parameters
INFO:flwr:Initializing global parameters
INFO :      Requesting initial parameters from one random client
INFO:flwr:Requesting initial parameters from one random client
INFO :      Sampling 1 clients, min None
INFO:flwr:Sampling 1 clients, min None


Number of available clients: 0
List of clients: {}
🚀 Starting Asynchronous Federated Learning Server...


INFO :      Sampling failed: number of available free clients (0) is less than number of requested free clients (1).
INFO:flwr:Sampling failed: number of available free clients (0) is less than number of requested free clients (1).


IndexError: list index out of range